# SPK-7 — Partition Pruning & Predicate Pushdown

**Break → Detect → Fix → Prove.** A filter on the partition column should let Spark read only the
matching partition and skip the rest. Wrap that column in a `CAST` (or a UDF) and Catalyst can no
longer match the filter to the partitions — so it reads **every** partition (a full scan).

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch the **SQL / DataFrame** tab's
scan node while the cells run.

**Laptop-safe:** this is the rare module that **writes a small table** — a few hundred thousand
rows across ~12 Parquet partitions under `.tmp/spk7_orders/` (a few MB). We only `count()` the
results, never collect them. The default **tuned** box is fine. **The last cell deletes the table**
(or run `make clean`).

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md) (SQL tab → scan node → `PartitionFilters`), and the
[troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import uniform_keys
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Pruning is decided at PLAN time — read it in SQL / DataFrame -> the Scan node.
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 — Parameters & a partitioned table on disk

To make pruning observable we need a **real partitioned table**. We generate a small fact with
`common.datagen.uniform_keys`, add an integer partition column `dt = pmod(row_id, 12)` (12 "day"
buckets), and write it as **Parquet partitioned by `dt`** under `.tmp/`. Then we read it back —
`spark.read.parquet(...)` rediscovers the partition layout, which is what pruning acts on.

In [ ]:
# Small on-disk table. Raise N_ROWS for a more dramatic full-scan vs pruned gap (keep it modest).
N_ROWS    = 240_000          # a few hundred k rows...
N_PARTS   = 12               # ...spread across 12 dt buckets (~20k rows each)
TABLE_DIR = ".tmp/spk7_orders"   # written under .tmp/ -> removed by teardown / `make clean`

apply_profile(spark, "tuned")    # default box; pruning is a plan-time concern, not a memory one

# Generate lazily, stamp a dt partition column, write partitioned Parquet (the only heavy step).
orders_src = (
    uniform_keys(spark, n_rows=N_ROWS, n_keys=1_000, key_col="customer_id")
    .withColumn("dt", F.pmod(F.col("row_id"), F.lit(N_PARTS)).cast("int"))
)
(orders_src.write.mode("overwrite").partitionBy("dt").parquet(TABLE_DIR))
print(f"wrote ~{N_ROWS:,} rows to {TABLE_DIR}/ partitioned by dt into {N_PARTS} buckets")

In [ ]:
# Read it back: Spark rediscovers dt as a partition column from the directory layout.
orders = spark.read.parquet(TABLE_DIR)
print("schema (note dt is the partition column):")
orders.printSchema()

# Confirm ~even rows per partition (small result, safe to show):
orders.groupBy("dt").count().orderBy("dt").show()

## 1. Break it — a `CAST` on the partition column

The dashboard passes the day as a string, so the "tidied up" query casts the column:
`WHERE CAST(dt AS STRING) = '5'`. That wraps the partition column in a function, so Catalyst can
no longer match it to the directory layout → **no pruning → all 12 partitions scanned.**

In [ ]:
# BROKEN: CAST the partition COLUMN -> defeats partition-filter matching.
broken = orders.where(F.col("dt").cast("string") == F.lit("5"))

m_broken = measure(spark, "broken: CAST(dt)='5' (full scan)", lambda: broken.count())
print("\nbroken metrics:", m_broken)

## 2. Detect it — read the plan

The headline detector is the **physical plan**, not a UI percentile. Read the `Scan parquet` node
of `broken.explain()` and look for **`PartitionFilters`**: it's **empty** `[]` here, meaning Spark
opened every partition. (Live equivalent: http://localhost:4040 → **SQL / DataFrame** → this
query → the **Scan parquet** node → its output rows ≈ the table total.)

See [`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md) → *SQL / DataFrame → scan node*.

In [ ]:
# Look for "PartitionFilters: []" (empty) in the Scan parquet node below = NO pruning.
broken.explain()

## 3. Diagnose

Partition pruning is a **plan-time** optimization: Catalyst matches predicates shaped like
`<partition_col> <op> <literal>` against the partition directories and skips the ones that can't
match — those files are never opened. Wrapping the column in **any** function (`CAST`, `substr`, a
UDF) yields an expression Catalyst can't invert against the partition values, so it conservatively
keeps **all** partitions and filters after scanning. The query stays *correct*; it just reads 12×
the data. The same trap forfeits **predicate pushdown to Parquet** on data columns — only simple
comparisons reach the row-group filter.

## 4. Fix it — filter the partition column directly

Compare the **raw** `dt` column to a **matching-typed literal** (`dt == 5`, an int). Catalyst now
sees a clean `dt = 5` predicate, populates `PartitionFilters`, and opens **only that one
directory** (~1/12 the data). If your input is a string, cast the *literal* — never the column:
`dt == F.lit('5').cast('int')`.

In [ ]:
# FIXED: compare the bare partition column to a matching-typed (int) literal -> pruning matches.
fixed = orders.where(F.col("dt") == F.lit(5))

m_fixed = measure(spark, "fixed: dt = 5 (pruned)", lambda: fixed.count())
print("\nfixed metrics:", m_fixed)

In [ ]:
# Now look for a populated "PartitionFilters: [isnotnull(dt), (dt = 5)]" = pruning ON,
# and a "PushedFilters" entry where data-column predicates push into Parquet.
fixed.explain()

## 4b. Prove it

Before/after. The full scan reads every row to answer the query; the pruned scan reads ~1/12 of
them for the **same** count — so runtime falls and the scan touches a single partition. (The
`Tasks` row also drops: fewer files means fewer scan tasks.)

In [ ]:
# Same answer, far less I/O. Pruned count == full-scan count (correctness preserved):
print(f"rows where dt==5  (broken full scan): {broken.count():,}")
print(f"rows where dt==5  (fixed pruned)    : {fixed.count():,}")
print(f"that is ~1/{N_PARTS} of the {N_ROWS:,}-row table\n")

compare([m_broken, m_fixed])

## Takeaways & "in real production..."

- **Detect** a pruning failure in the **plan**, not by feel: an **empty `PartitionFilters`** (or a
  scan whose output rows ≈ the whole table) where you expected one partition.
- **Never apply a function to the column you prune/filter on.** Cast or transform the **literal**
  side instead, or store the partition column in the type you query it with.
- The same rule governs **predicate pushdown** to Parquet/ORC data columns: simple `col <op>
  literal` / `IN` / `BETWEEN` push down (`PushedFilters`); UDFs and wrapped columns do not.
- **In prod:** make `explain()` / the SQL-tab scan node part of code review for hot partitioned
  queries; alert when a job's scanned bytes ≈ full-table size despite a partition filter.

## 5. Teardown

This module **wrote a table** under `.tmp/`, so unlike the count-only modules there *is* something
to delete. The cell below removes `.tmp/spk7_orders/` and restores the `tuned` profile. If anything
is left behind, `make clean` clears everything under `.tmp/`.

In [ ]:
import shutil

shutil.rmtree(TABLE_DIR, ignore_errors=True)   # remove the small partitioned table we wrote
apply_profile(spark, "tuned")                  # restore production-tuned safety nets
spark.catalog.clearCache()
print(f"Done. Removed {TABLE_DIR}/ and reset profile to 'tuned'. `make clean` clears .tmp if needed.")